In [ ]:
import torch
from torch import nn
from d2l import torch as d2l

In [ ]:
class BatchNorm(nn.Module):
    def __init__(self, num_feature, num_dim):
        assert num_dim in (2, 4)
        super().__init__()
        if num_dim == 2:
            para_shape = (1, num_feature)
        elif num_dim == 4:
            para_shape = (1, num_feature, 1, 1)
        self.gamma = nn.Parameter(torch.ones(para_shape))
        self.beta = nn.Parameter(torch.zeros(para_shape))
        self.moving_mean = torch.zeros(para_shape)
        self.moving_var = torch.ones(para_shape)

    def forward(self, X):
        # 如果X不在内存上，将 moving_mean 和 moving_var
        # 复制到X所在显存上
        if self.moving_mean.device != X.device:
            self.moving_mean = self.moving_mean.to(X.device)
            self.moving_var = self.moving_var.to(X.device)

        Y, self.moving_mean, self.moving_var = BatchNorm.batch_norm(X, self.gamma, self.beta, 1e-5, self.moving_mean, self.moving_var, 0.1)

        return Y
    
    #归一化函数
    # X: 输入矩阵    gamma，beta：可学习参数     eps：为了防止方差为 0 的安全保护项
    # moving_mean，moving_var：全局滑动算术平均值和方差（非单个样本的）
    # momentum：滑动平均值的加权参数
    @classmethod
    def batch_norm(X, gamma, beta, eps, moving_mean, moving_var, momentum):
        #如果不是训练模式，而是推理模式，那么要采用全局的均值和方差计算
        if not torch.is_grad_enabled():
            X_hat = (X - moving_mean) / torch.sqrt(moving_var + eps)
        else:
            #检验输入特征的正确维度：
            #如果是全连接层就是（1 * dim）
            #如果是卷积层就是（batch_size * channel * hignt * width）
            num_dim = len(X.shape)
            assert num_dim in (2, 4)
            if num_dim == 2:
                #如果是全连接层的输入或输出，那么就对竖直方向压缩求平均、方差即可（结果是一维的）
                mean = X.mean(dim = 0)
                var = ((X - mean) ** 2).mean(dim = 0)
            elif num_dim == 4:
                #如果是卷积层，那么对除通道数所在维度的其他方向压缩求平均和方差（仍然是四维）
                mean = X.mean(dim = (0, 2, 3), keepdim = True)
                var = ((X - mean) ** 2).mean(dim = (0, 2, 3), keepdim = True)
            #训练模式，采用样本的均值和方差计算归一化的 X_hat
            X_hat = (X - mean) / torch.sqrt(var + eps)
            #更新滑动均值，这里使用权重来更新是为了是过渡平滑并且具有遗忘
            moving_mean = (1.0 - momentum) * moving_mean + momentum * mean
            moving_var = (1.0 - momentum) * moving_var + momentum * var
        #输出学习后的归一化值
        Y = gamma * X_hat + beta
        return Y, moving_mean, moving_var    